# 01b — Category Taxonomy
Apply the multi-label category taxonomy from `src.data.target_engineering`.

In [7]:
import os
from pathlib import Path

# Find project root (contains pyproject.toml)
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

Working directory: e:\OSFDA


In [8]:
import pandas as pd

from src.utils.config import load_main_config, resolve_path
from src.data.loader import load_raw_data
from src.data.target_engineering import apply_category_taxonomy

config = load_main_config()
df = load_raw_data(config)
print(f'Loaded {len(df)} records.')

Loaded 38655 records from E:\OSFDA\data\raw\asrs_full.parquet
Loaded 38655 records.


## 1. Apply Category Taxonomy

In [9]:
df, cat_matrix = apply_category_taxonomy(df)

print('Category counts:')
print(cat_matrix.sum())
print(f'\nAverage labels/report: {cat_matrix.sum(axis=1).mean():.2f}')
zero = (cat_matrix.sum(axis=1) == 0).sum()
print(f'Zero-label rows: {zero} ({zero/len(df)*100:.1f}%)')

Category counts:
ATC_Communication       6592
Airspace_Navigation     2282
Environment             8510
Equipment_System       16439
Flight_Operations      28648
dtype: int64

Average labels/report: 1.62
Zero-label rows: 139 (0.4%)


## 2. Primary Category Distribution

In [10]:
print('Primary category distribution:')
print(df['primary_category'].value_counts())

Primary category distribution:
primary_category
Flight_Operations      19517
Equipment_System       10731
Other                   3611
Environment             2455
ATC_Communication       1754
Airspace_Navigation      587
Name: count, dtype: int64


## 3. Save Category Targets

In [11]:
cat_path = resolve_path('data/processed/category_targets.parquet')
output_df = pd.concat([df[['acn_num_ACN', 'primary_category']], cat_matrix], axis=1)
output_df.to_parquet(cat_path, index=False)
print(f'Category targets saved to {cat_path}')

Category targets saved to E:\OSFDA\data\processed\category_targets.parquet
